# 통제형 Supervisor 에이전트

Supervisor가 전문 에이전트를 순서대로 호출한다. 마지막에 사람이 결과물을 승인한다.


Supervisor는 전체 흐름을 통제합니다. Request/Planner가 계획을 세우고 Retrieval이 자료를 모읍니다. Analysis가 정리하고 Writer가 초안을 씁니다. Reviewer가 본 뒤 Human Approval에서 멈춥니다.

- Request/Planner Agent: 요청을 분석하고 작업 계획을 세웁니다.
- Retrieval Agent: 계획에 맞춰 관련 정보를 모읍니다.
- Analysis Tool Agent: 모은 정보를 분석하고 요약합니다.
- Writer Agent: 분석 결과로 초안을 씁니다.
- Reviewer Agent: 초안을 검토합니다.
- Human Approval Node: 사람이 승인하거나 다시 쓰게 합니다.


## 1. 패키지


In [ ]:
%pip install -q langchain langchain-openai langgraph pydantic


## 2. 런타임


In [ ]:
from typing import List
from typing_extensions import Literal, TypedDict
from pydantic import BaseModel, Field
from langchain.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from IPython.display import Image, display
from pathlib import Path
import json
import re
import sys

if sys.stdout.encoding and sys.stdout.encoding.lower().replace("-", "") != "utf8":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")


## 3. 스키마

Planner는 단계와 검색 질의를 내고, Reviewer는 피드백과 승인 여부를 냅니다.


In [ ]:
class Plan(BaseModel):
    """Request/Planner Agent가 생성하는 작업 계획

    주의: 중첩 객체 구조(PlanStep)는 gpt-oss 모델에서 구조화된 출력이
    불안정하므로, 단순한 List[str] 구조를 사용합니다.
    각 step 문자열은 "작업내용 (핵심: 키워드1, 키워드2)" 형식으로 작성합니다.
    """
    steps: List[str] = Field(
        description=(
            "사용자 요청을 처리하기 위한 순차적 작업 계획 (최소 2개, 최대 6개). "
            '각 단계는 "작업내용 (핵심: 키워드1, 키워드2)" 형식의 문자열입니다. '
            '예: "사용자 요청의 핵심 주제와 목적을 분석한다 (핵심: 요청 분석, 핵심 주제)"'
        ),
        min_length=2,
        max_length=6,
    )
    search_queries: List[str] = Field(
        description=(
            "Retrieval Agent가 사용할 검색 질의 목록 (최소 2개, 최대 5개). "
            "각 질의는 검색 엔진에 입력할 수 있는 명확한 한국어 문장 또는 키워드 조합입니다. "
            '예: "AI 에이전트 발전 과정"'
        ),
        min_length=2,
        max_length=5,
    )


class ReviewResult(BaseModel):
    """Reviewer Agent가 생성하는 검토 결과"""
    feedback: str = Field(description="초안에 대한 피드백/개선점")
    approved: bool = Field(description="초안이 품질 기준을 통과했는지 여부")


## 4. LLM

로컬 LM Studio 엔드포인트에 ChatOpenAI를 붙입니다. Planner와 Reviewer는 `json_schema`로 구조화된 답을 받습니다.


In [ ]:
llm = ChatOpenAI(
    base_url="http://[your_host_address]:1234/v1",
    api_key="mypassword", 
    model="google/gemma-4-26b-a4b",
)

planner_llm = llm.with_structured_output(Plan, method="json_schema")
reviewer_llm = llm.with_structured_output(ReviewResult, method="json_schema")


## 5. 그래프 상태

노드가 읽고 쓰는 칸입니다. `revision_count`로 재작성 횟수를 셉니다.


In [ ]:
class State(TypedDict):
    input: str
    plan: str
    search_queries: List[str]
    retrieved_docs: str
    analysis: str
    draft: str
    review_feedback: str
    review_approved: bool
    human_approved: bool
    final_output: str
    revision_count: int


## 6. 에이전트 노드

Planner 응답에서 빈 문자열을 걸러 냅니다. 구조화된 출력이 비면 텍스트에서 JSON을 찾습니다. 그래도 안 되면 기본 계획을 씁니다.


In [ ]:
_GARBAGE_PATTERNS = [
    re.compile(r"\.{3,}"),
    re.compile(r"\?{2,}"),
    re.compile(r"-{3,}"),
    re.compile(r"^[\s\W]+$"),
    re.compile(r'["\']{2,}'),
]


def _is_garbage(text: str) -> bool:
    if not text or not text.strip():
        return True
    if len(re.findall(r"[가-힣a-zA-Z]", text)) < 3:
        return True
    for pattern in _GARBAGE_PATTERNS:
        if pattern.search(text):
            return True
    return False


def _parse_json_from_text(text: str) -> dict | None:
    json_block = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL)
    if json_block:
        try:
            return json.loads(json_block.group(1))
        except json.JSONDecodeError:
            pass
    brace_block = re.search(r"\{.*\}", text, re.DOTALL)
    if brace_block:
        try:
            return json.loads(brace_block.group(0))
        except json.JSONDecodeError:
            pass
    return None


### Request/Planner Agent


In [ ]:
def request_planner_agent(state: State):
    system_prompt = (
        "당신은 요청 분석 및 계획 수립 전문가입니다.\n"
        "사용자의 요청을 분해하여 순차적으로 수행 가능한 작업 계획을 세우고, "
        "정보 검색에 사용할 검색 질의를 생성하세요.\n\n"
        "[출력 규칙]\n"
        "1. steps: 최소 2개, 최대 6개의 단계를 작성하세요.\n"
        '   각 단계는 "작업내용 (핵심: 키워드1, 키워드2)" 형식의 문자열입니다.\n'
        "   작업내용은 완결된 한국어 문장(10자 이상)으로 작성하세요.\n"
        "2. search_queries: 최소 2개, 최대 5개의 검색 질의를 작성하세요.\n"
        "   각 질의는 검색 엔진에 입력할 수 있는 명확한 한국어 문장 또는 키워드 조합이어야 합니다.\n"
        "   빈 문자열이나 의미 없는 문자(..., ???, ---)를 포함하지 마세요.\n\n"
        "[출력 예시]\n"
        "steps:\n"
        '  - "사용자 요청의 핵심 주제와 목적을 분석한다 (핵심: 요청 분석, 핵심 주제)"\n'
        '  - "관련 정보를 검색하여 수집한다 (핵심: 정보 검색, 자료 수집)"\n'
        "search_queries:\n"
        '  - "AI 에이전트 발전 과정"\n'
        '  - "AI 에이전트 향후 전망"'
    )

    max_retries = 3
    result = None
    last_error = None

    for attempt in range(max_retries):
        try:
            result = planner_llm.invoke(
                [
                    SystemMessage(content=system_prompt),
                    HumanMessage(content=f"사용자 요청: {state['input']}"),
                ]
            )
            valid_steps = [s for s in result.steps if s and s.strip() and not _is_garbage(s)]
            valid_queries = [q for q in result.search_queries if q and q.strip() and not _is_garbage(q)]
            if len(valid_steps) >= 2 and len(valid_queries) >= 2:
                result.steps = valid_steps
                result.search_queries = valid_queries
                break
            last_error = (
                f"구조화된 출력 검증 실패 (시도 {attempt + 1}/{max_retries}): "
                f"유효 steps={len(valid_steps)}/{len(result.steps)}, "
                f"유효 queries={len(valid_queries)}/{len(result.search_queries)}"
            )
            print(f"[Planner] {last_error} → 재시도")
            bad_steps = [s for s in result.steps if not s or not s.strip() or _is_garbage(s)]
            bad_queries = [q for q in result.search_queries if not q or not q.strip() or _is_garbage(q)]
            if bad_steps:
                print(f"  [거부된 steps] {bad_steps}")
            if bad_queries:
                print(f"  [거부된 queries] {bad_queries}")
            result = None
        except Exception as e:
            last_error = str(e)
            print(f"[Planner] 구조화된 출력 실패 (시도 {attempt + 1}/{max_retries}): {e} → 재시도")
            result = None

    if result is None:
        print("[Planner] 구조화된 출력 실패 → 일반 텍스트 응답에서 JSON 파싱 시도")
        text_prompt = (
            f"{system_prompt}\n\n"
            "반드시 아래 JSON 형식으로만 응답하세요. 다른 설명은 하지 마세요.\n"
            "```json\n"
            "{\n"
            '  "steps": ["단계1 (핵심: 키워드)", "단계2 (핵심: 키워드)"],\n'
            '  "search_queries": ["검색어1", "검색어2"]\n'
            "}\n"
            "```"
        )
        try:
            msg = llm.invoke(
                [
                    SystemMessage(content=text_prompt),
                    HumanMessage(content=f"사용자 요청: {state['input']}"),
                ]
            )
            parsed = _parse_json_from_text(msg.content)
            if parsed and "steps" in parsed and "search_queries" in parsed:
                valid_steps = [s for s in parsed["steps"] if s and s.strip() and not _is_garbage(s)]
                valid_queries = [q for q in parsed["search_queries"] if q and q.strip() and not _is_garbage(q)]
                if len(valid_steps) >= 2 and len(valid_queries) >= 2:
                    result = Plan(steps=valid_steps, search_queries=valid_queries)
                    print("[Planner] JSON 파싱 성공!")
        except Exception as e:
            last_error = str(e)
            print(f"[Planner] JSON 파싱 실패: {e}")

    if result is None or len(result.steps) < 2 or len(result.search_queries) < 2:
        print(f"[Planner] 모든 시도 실패. 폴백 계획 사용. 마지막 오류: {last_error}")
        user_input = state["input"]
        fallback_steps = [
            "사용자 요청을 분석하여 핵심 주제와 목적을 파악한다 (핵심: 요청 분석, 핵심 주제)",
            "핵심 주제에 대한 관련 정보를 검색하고 수집한다 (핵심: 정보 검색, 자료 수집)",
            "수집된 정보를 분석하여 핵심 인사이트를 도출한다 (핵심: 정보 분석, 인사이트)",
            "분석 결과를 바탕으로 최종 결과물을 작성한다 (핵심: 결과 작성, 보고서)",
        ]
        fallback_queries = [
            user_input[:100].strip() or user_input,
            user_input[:50].strip() or user_input,
        ]
        result = Plan(steps=fallback_steps, search_queries=fallback_queries)

    plan_text = "\n".join(f"  {i + 1}. {s}" for i, s in enumerate(result.steps))
    return {
        "plan": plan_text,
        "search_queries": result.search_queries,
    }


### Retrieval Agent


In [ ]:
def retrieval_agent(state: State):
    queries = state.get("search_queries", [])
    queries_text = "\n".join(f"  - {q}" for q in queries)
    msg = llm.invoke(
        [
            SystemMessage(
                content=(
                    "당신은 정보 검색 에이전트입니다. "
                    "주어진 검색 질의들에 대해 관련성 높은 정보를 수집하여 정리하세요. "
                    "실제 검색 대신, 당신의 지식을 기반으로 각 질의에 대한 핵심 정보를 제공하세요."
                )
            ),
            HumanMessage(
                content=(
                    f"작업 계획:\n{state['plan']}\n\n"
                    f"검색 질의 목록:\n{queries_text}\n\n"
                    "각 질의에 대한 검색 결과를 정리해주세요."
                )
            ),
        ]
    )
    return {"retrieved_docs": msg.content}


### Analysis Tool Agent


In [ ]:
def analysis_tool_agent(state: State):
    msg = llm.invoke(
        [
            SystemMessage(
                content=(
                    "당신은 데이터 분석 에이전트입니다. "
                    "검색된 정보를 분석하여 핵심 인사이트, 패턴, 통찰을 도출하고, "
                    "작성자가 보고서를 작성하는 데 활용할 수 있도록 정리하세요."
                )
            ),
            HumanMessage(
                content=(
                    f"원본 요청: {state['input']}\n\n"
                    f"작업 계획:\n{state['plan']}\n\n"
                    f"검색된 정보:\n{state['retrieved_docs']}\n\n"
                    "이 정보를 분석하여 핵심 인사이트를 도출해주세요."
                )
            ),
        ]
    )
    return {"analysis": msg.content}


### Writer Agent


In [ ]:
def writer_agent(state: State):
    feedback = state.get("review_feedback", "")
    revision_count = state.get("revision_count", 0)
    system_prompt = (
        "당신은 전문 작성 에이전트입니다. "
        "분석 결과를 바탕으로 사용자의 요청에 부합하는 명확하고 체계적인 결과물을 작성하세요."
    )
    user_content = (
        f"원본 요청: {state['input']}\n\n"
        f"작업 계획:\n{state['plan']}\n\n"
        f"분석 결과:\n{state['analysis']}\n\n"
    )
    if feedback and revision_count > 0:
        user_content += (
            f"\n[이전 초안에 대한 검토 피드백 - 반드시 반영하여 개선하세요]\n"
            f"{feedback}\n\n"
            f"위 피드백을 반영하여 개선된 결과물을 작성하세요. (재작성 {revision_count}회차)"
        )
    msg = llm.invoke(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_content),
        ]
    )
    return {"draft": msg.content}


### Reviewer Agent


In [ ]:
def reviewer_agent(state: State):
    result = reviewer_llm.invoke(
        [
            SystemMessage(
                content=(
                    "당신은 엄격한 검토자 에이전트입니다. "
                    "초안이 사용자의 원본 요청을 충족하는지, 내용이 정확하고 체계적인지 평가하세요. "
                    "명백한 결함이 없으면 approved=true로 설정하고, 개선이 필요하면 "
                    "approved=false와 함께 구체적인 피드백을 제공하세요."
                )
            ),
            HumanMessage(
                content=(
                    f"원본 요청: {state['input']}\n\n"
                    f"초안:\n{state['draft']}\n\n"
                    "초안을 검토하고 승인 여부를 결정해주세요."
                )
            ),
        ]
    )
    return {
        "review_feedback": result.feedback,
        "review_approved": result.approved,
    }


### Human Approval Node

`interrupt()`에서 실행이 멈춥니다. 다음 셀에서 `Command(resume="approve")` 또는 `reject`로 이어갑니다.


In [ ]:
def human_approval_node(state: State):
    print("\n" + "=" * 60)
    print("[Human Approval Node] 최종 결과물 검토")
    print("=" * 60)
    print(f"\n[원본 요청]\n{state['input']}")
    print(f"\n[작성된 결과물]\n{state['draft']}")
    print("\n" + "=" * 60)

    user_decision = interrupt(
        {
            "question": "이 결과물을 승인하시겠습니까?",
            "draft": state["draft"],
            "options": ["approve", "reject"],
        }
    )
    approved = user_decision == "approve"
    return {
        "human_approved": approved,
        "final_output": state["draft"] if approved else "",
    }


## 7. Supervisor 라우팅

검토가 통과하면 사람 승인으로 갑니다. 실패하면 Writer로 돌아갑니다. 재작성은 최대 3회입니다.


In [ ]:
def route_after_review(state: State) -> Literal["human_approval", "writer"]:
    if state.get("review_approved", False):
        return "human_approval"
    if state.get("revision_count", 0) >= 3:
        print("\n[Supervisor] 최대 재작성 횟수(3회) 도달 → Human Approval로 강제 진행")
        return "human_approval"
    print(f"\n[Supervisor] 검토 미통과 → Writer Agent로 재작성 지시 ({state.get('revision_count', 0) + 1}회차)")
    return "writer"


def route_after_human(state: State) -> Literal[END, "writer"]:
    if state.get("human_approved", False):
        return END
    revision_count = state.get("revision_count", 0)
    if revision_count >= 5:
        print("\n[Supervisor] 사용자 거부 횟수(5회) 초과 → 현재 결과물로 종료")
        return END
    print(f"\n[Supervisor] 사용자 거부 → Writer Agent로 재작성 지시 ({revision_count + 1}회차)")
    return "writer"


def increment_revision(state: State):
    return {"revision_count": state.get("revision_count", 0) + 1}


## 8. 그래프

노드를 잇고 `MemorySaver`로 interrupt를 저장합니다.


In [ ]:
supervisor_builder = StateGraph(State)

supervisor_builder.add_node("request_planner", request_planner_agent)
supervisor_builder.add_node("retrieval", retrieval_agent)
supervisor_builder.add_node("analysis", analysis_tool_agent)
supervisor_builder.add_node("increment_revision", increment_revision)
supervisor_builder.add_node("writer", writer_agent)
supervisor_builder.add_node("reviewer", reviewer_agent)
supervisor_builder.add_node("human_approval", human_approval_node)

supervisor_builder.add_edge(START, "request_planner")
supervisor_builder.add_edge("request_planner", "retrieval")
supervisor_builder.add_edge("retrieval", "analysis")
supervisor_builder.add_edge("analysis", "increment_revision")
supervisor_builder.add_edge("increment_revision", "writer")
supervisor_builder.add_edge("writer", "reviewer")

supervisor_builder.add_conditional_edges(
    "reviewer",
    route_after_review,
    {
        "human_approval": "human_approval",
        "writer": "increment_revision",
    },
)
supervisor_builder.add_conditional_edges(
    "human_approval",
    route_after_human,
    {
        END: END,
        "writer": "increment_revision",
    },
)

checkpointer = MemorySaver()
supervisor_workflow = supervisor_builder.compile(checkpointer=checkpointer)


## 9. 다이어그램


In [ ]:
png = supervisor_workflow.get_graph().draw_mermaid_png()
Path("Supervisor.png").write_bytes(png)
display(Image(data=png))
print("saved Supervisor.png")


## 10. 실행

첫 셀은 Human Approval에서 멈춥니다. 그다음 셀에서 승인합니다.


In [ ]:
initial_state = {
    "input": "AI 에이전트의 발전 과정과 향후 전망에 대한 보고서를 작성해줘",
    "revision_count": 0,
}

config = {"configurable": {"thread_id": "supervisor-demo"}}
result = supervisor_workflow.invoke(initial_state, config=config)

print("draft 길이", len(result.get("draft", "")), "자")
print("review_approved", result.get("review_approved"))


In [ ]:
final_result = supervisor_workflow.invoke(Command(resume="approve"), config=config)

print("사용자 승인 여부:", final_result.get("human_approved"))
print("재작성 횟수:", final_result.get("revision_count"))
print()
print(final_result.get("final_output", final_result.get("draft", "")))
